### DATA TRAINER MODULE

### Workflows


1. **Config.yaml**: Este archivo contendrá toda la configuración relacionada con el proyecto. Se utilizará para almacenar todos los parámetros relacionados con el proyecto, así como todas las rutas del proyecto. También se usará para guardar todos los hiperparámetros.
2. **Params.yaml**: Este archivo contendrá todos los hiperparámetros relacionados con el proyecto. Se utilizará para almacenar los hiperparámetros del proyecto y, específicamente, los hiperparámetros del entrenamiento del modelo.
3. **Entidad de configuración (Config entity)**: Este componente definirá la estructura de los archivos de configuración y proporcionará una forma de acceder a los parámetros de configuración de manera estructurada.
4. **Administrador de configuración (Configuration Manager)**: Este componente gestionará la carga y actualización de los archivos de configuración, asegurando que se utilice la configuración correcta en todo el proyecto.
5. **Actualizar componentes**: Ingesta de datos, Transformación de datos, Entrenador del modelo.
6. **Crear pipelines**: Pipeline de entrenamiento, Pipeline de predicción.

In [1]:
import os
%pwd

'c:\\Users\\mecenturion\\Desktop\\MARTIN\\UDEMY\\MLOPS\\TextSummarizer\\textsummarizer\\research'

In [2]:
os.chdir('..')
%pwd

'c:\\Users\\mecenturion\\Desktop\\MARTIN\\UDEMY\\MLOPS\\TextSummarizer\\textsummarizer'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [4]:
from src.textSummarizer.utils.common import read_yaml, create_directories
from src.textSummarizer.constants import *
from datasets import load_from_disk


# Creamos el configuration mangarer, que se encargará de cargar el archivo de configuración y devolver la configuración de cada componente.
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        

        # Creamos el directorio de Artifacts, que es donde se guardarán todos los artefactos del proyecto.
        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])
    
        model_trainer_config = ModelTrainerConfig(
            root_dir = Path(config.root_dir),
            data_path = Path(config.data_path),
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            per_device_eval_batch_size = params.per_device_eval_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )
        
        return model_trainer_config

c:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:

from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_from_disk
import torch


class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        

        
    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        #Training
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
        dataset = load_from_disk(self.config.data_path)
        trainer_args = TrainingArguments(
            output_dir = self.config.root_dir, num_train_epochs = self.config.num_train_epochs, warmup_steps = self.config.warmup_steps,
            per_device_train_batch_size = self.config.per_device_train_batch_size, per_device_eval_batch_size = self.config.per_device_eval_batch_size,
            weight_decay = self.config.weight_decay, logging_steps = self.config.logging_steps,
            eval_strategy=self.config.evaluation_strategy, eval_steps = self.config.eval_steps, save_steps = int(float(self.config.save_steps)),
            gradient_accumulation_steps = self.config.gradient_accumulation_steps
        )
        
        trainer = Trainer(
            model=model,
            args=trainer_args,
            train_dataset=dataset["train"].select(range(300)),
            eval_dataset=dataset["validation"],
            data_collator=seq2seq_data_collator
        )
        trainer.train()
        #Save the model
        trainer.save_model(os.path.join(self.config.root_dir, "pegasus_samsum_model"))
        #Save the tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))

In [16]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(model_trainer_config)
model_trainer.train()

[2026-04-20 11:37:25,982: INFO: common: 31: YAML file 'C:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\config\config.yaml' read successfully.]
[2026-04-20 11:37:25,995: INFO: common: 31: YAML file 'C:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\params.yaml' read successfully.]
[2026-04-20 11:37:25,999: INFO: common: 53: Directory 'artifacts' created successfully.]
[2026-04-20 11:37:26,002: INFO: common: 53: Directory 'artifacts/model_trainer' created successfully.]
[2026-04-20 11:37:26,245: INFO: _client: 1025: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-04-20 11:37:26,272: INFO: _client: 1025: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-04-20 11:37:26,439: INFO: _client: 1025: HTTP Request: HEAD https://hu

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 27194.71it/s]
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[2026-04-20 11:37:32,765: INFO: _client: 1025: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-04-20 11:37:32,827: INFO: _client: 1025: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK"]


Epoch,Training Loss,Validation Loss
1,148.996045,9.105177


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.77s/it]
c:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.02s/it]
